In [ ]:
!pip install git+https://github.com/d2l-ai/d2l-zh@release  # installing d2l

In [ ]:
import torch
from torch import nn

def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    kh, kw = K.shape
    nh, nw = X.shape
    print(f"kh, kw {kh, kw}, nh, nw {nh, nw}")
    Y = torch.zeros((nh - kh + 1, nw - kw + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i : i + kh, j : j + kw] * K).sum()
    return Y

class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

X = torch.ones((6, 8))
X[:, 2:6] = 0
print(X)
print(f'Input shape: {X.shape}')

K = torch.tensor([[-1.0, 1.0]]) # 手动设置卷积核
print(f'Kernel shape: {K.shape}')

Y = corr2d(X, K)
print(f'Output shape: {Y.shape}')
# print(corr2d(X.T, K))




tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
Input shape: torch.Size([6, 8])
Kernel shape: torch.Size([1, 2])
kh, kw (1, 2), nh, nw (6, 8)
Output shape: torch.Size([6, 7])


In [4]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False) 
# 利用nn.T_destination构造卷积层,而非手动设置卷积核
# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、(高度, 宽度)、bias），
inputs = X.reshape((1, 1, 6, 8))
outputs = Y.reshape((1, 1, 6, 7))
lr = 6*7*3e-3 
loss = nn.MSELoss()

for i in range(1000):
    Y_hat = conv2d(inputs)   # 1. 前向传播
    l = loss(Y_hat, outputs) # 2. 计算损失
    conv2d.zero_grad()       # 3. 清除梯度
    l.sum().backward()       # 4. 计算梯度
    conv2d.weight.data[:] -= lr * conv2d.weight.grad # 5. 更新权重
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

print('conv2d:', conv2d)
print('conv2d.weight:', conv2d.weight)
print('conv2d.weight.data:', conv2d.weight.data)
print('conv2d.weight.data.shape:', conv2d.weight.data.shape)
print('conv2d.weight.data.reshape((1, 2)):', conv2d.weight.data.reshape((1, 2)))




epoch 2, loss 0.575
epoch 4, loss 0.473
epoch 6, loss 0.398
epoch 8, loss 0.339
epoch 10, loss 0.291
epoch 12, loss 0.250
epoch 14, loss 0.215
epoch 16, loss 0.186
epoch 18, loss 0.160
epoch 20, loss 0.138
epoch 22, loss 0.120
epoch 24, loss 0.103
epoch 26, loss 0.089
epoch 28, loss 0.077
epoch 30, loss 0.067
epoch 32, loss 0.057
epoch 34, loss 0.050
epoch 36, loss 0.043
epoch 38, loss 0.037
epoch 40, loss 0.032
epoch 42, loss 0.028
epoch 44, loss 0.024
epoch 46, loss 0.021
epoch 48, loss 0.018
epoch 50, loss 0.015
epoch 52, loss 0.013
epoch 54, loss 0.011
epoch 56, loss 0.010
epoch 58, loss 0.009
epoch 60, loss 0.007
epoch 62, loss 0.006
epoch 64, loss 0.005
epoch 66, loss 0.005
epoch 68, loss 0.004
epoch 70, loss 0.004
epoch 72, loss 0.003
epoch 74, loss 0.003
epoch 76, loss 0.002
epoch 78, loss 0.002
epoch 80, loss 0.002
epoch 82, loss 0.001
epoch 84, loss 0.001
epoch 86, loss 0.001
epoch 88, loss 0.001
epoch 90, loss 0.001
epoch 92, loss 0.001
epoch 94, loss 0.001
epoch 96, loss 0.

In [8]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、(高度, 宽度)、bias），
# 其中批量大小和通道数都为1
inputs = X.reshape((1, 1, 6, 8))
outputs = Y.reshape((1, 1, 6, 7))
lr = 6*7*3e-3 
loss = nn.MSELoss()

optimizer = torch.optim.SGD(conv2d.parameters(), lr=lr)   # 循环外，建一次 optimizer
for i in range(100):
    Y_hat = conv2d(inputs)              # 1. 前向
    l = loss(Y_hat, outputs)            # 2. 算损失
    optimizer.zero_grad()               # 3. 清梯度（在 backward 之前！）
    l.sum().backward()                  # 4. 反向，算梯度
    optimizer.step()                    # 5. 更新（在 backward 之后！）
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 0.286
epoch 4, loss 0.215
epoch 6, loss 0.172
epoch 8, loss 0.142
epoch 10, loss 0.119
epoch 12, loss 0.102
epoch 14, loss 0.087
epoch 16, loss 0.075
epoch 18, loss 0.065
epoch 20, loss 0.056
epoch 22, loss 0.048
epoch 24, loss 0.042
epoch 26, loss 0.036
epoch 28, loss 0.031
epoch 30, loss 0.027
epoch 32, loss 0.023
epoch 34, loss 0.020
epoch 36, loss 0.017
epoch 38, loss 0.015
epoch 40, loss 0.013
epoch 42, loss 0.011
epoch 44, loss 0.010
epoch 46, loss 0.008
epoch 48, loss 0.007
epoch 50, loss 0.006
epoch 52, loss 0.005
epoch 54, loss 0.005
epoch 56, loss 0.004
epoch 58, loss 0.003
epoch 60, loss 0.003
epoch 62, loss 0.003
epoch 64, loss 0.002
epoch 66, loss 0.002
epoch 68, loss 0.002
epoch 70, loss 0.001
epoch 72, loss 0.001
epoch 74, loss 0.001
epoch 76, loss 0.001
epoch 78, loss 0.001
epoch 80, loss 0.001
epoch 82, loss 0.001
epoch 84, loss 0.001
epoch 86, loss 0.000
epoch 88, loss 0.000
epoch 90, loss 0.000
epoch 92, loss 0.000
epoch 94, loss 0.000
epoch 96, loss 0.